# AI-Based Phishing URL Detection — STEP 1: Dataset Exploration

Notebook ini adalah tempat kita mengerjakan project secara bertahap. Untuk sekarang kita **hanya fokus di STEP 1: memahami dataset mentah** — belum masuk ke feature engineering atau modeling.

## Tujuan STEP 1
1. Memuat dataset ke dalam notebook
2. Memahami struktur kolom & tipe data
3. Mengecek jumlah data
4. Mengecek distribusi label (Good vs Bad)
5. Mengecek missing values
6. Mengecek duplikat
7. Melihat contoh URL phishing vs legitimate secara langsung

**Cara pakai notebook ini:**
- Upload file `phishing_site_urls.csv` (hasil download dari Kaggle) ke folder yang sama dengan notebook ini (atau ke environment Colab kamu).
- Jalankan cell satu per satu dari atas ke bawah.
- Jangan skip ke bagian modeling — kita akan tambahkan itu di notebook terpisah setelah tahap ini benar-benar jelas.

## 1. Import Library

Untuk tahap eksplorasi awal, kita cuma butuh `pandas` untuk baca & inspeksi data, dan `numpy` untuk operasi numerik dasar. Library lain (scikit-learn, dll) belum diperlukan di tahap ini.

In [1]:
import pandas as pd
import numpy as np

## 2. Load Dataset

Ganti nama file di bawah ini jika nama file hasil download kamu berbeda (kadang Kaggle memberi nama seperti `phishing_site_urls.csv` di dalam file zip).

In [2]:
DATA_PATH = "../data/raw/phishing_site_urls.csv"  # ganti jika nama file berbeda

df = pd.read_csv(DATA_PATH)
df.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


## 3. Struktur Data

Cek nama kolom, tipe data, dan jumlah baris. Ini penting supaya kita tahu persis apa yang sedang kita kerjakan — jangan berasumsi dulu.

In [3]:
print("Jumlah baris & kolom:", df.shape)
print()
print("Tipe data tiap kolom:")
df.info()

Jumlah baris & kolom: (549346, 2)

Tipe data tiap kolom:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549346 entries, 0 to 549345
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   URL     549346 non-null  object
 1   Label   549346 non-null  object
dtypes: object(2)
memory usage: 8.4+ MB


## 4. Distribusi Label

Ini krusial: kalau labelnya sangat tidak seimbang (misalnya 90% Good, 10% Bad), itu akan memengaruhi cara kita split data nanti dan metrik evaluasi mana yang bisa dipercaya (accuracy jadi tidak cukup).

In [4]:
label_col = "Label"  # sesuaikan jika nama kolom label berbeda di dataset kamu

print(df[label_col].value_counts())
print()
print(df[label_col].value_counts(normalize=True) * 100)

Label
good    392924
bad     156422
Name: count, dtype: int64

Label
good    71.525778
bad     28.474222
Name: proportion, dtype: float64


## 5. Missing Values

Cek apakah ada baris dengan URL atau label kosong. Data yang hilang bisa merusak proses feature extraction nanti kalau tidak ditangani.

In [5]:
df.isnull().sum()

URL      0
Label    0
dtype: int64

## 6. Duplikat

URL yang duplikat bisa menyebabkan data leakage kalau baris yang sama muncul di train dan test set sekaligus. Kita cek dulu seberapa besar masalahnya.

In [6]:
url_col = "URL"  # sesuaikan jika nama kolom URL berbeda

print("Jumlah baris duplikat (seluruh kolom):", df.duplicated().sum())
print("Jumlah URL duplikat (kolom URL saja):", df.duplicated(subset=[url_col]).sum())

Jumlah baris duplikat (seluruh kolom): 42150
Jumlah URL duplikat (kolom URL saja): 42151


## 7. Contoh Data per Kelas

Lihat langsung beberapa contoh URL phishing dan legitimate — ini membantu kita membangun intuisi sebelum masuk ke feature engineering di notebook berikutnya.

In [7]:
print("Contoh URL Bad (phishing):")
display(df[df[label_col] == "bad"][url_col].head(10))

print("\nContoh URL Good (legitimate):")
display(df[df[label_col] == "good"][url_col].head(10))

Contoh URL Bad (phishing):


0    nobell.it/70ffb52d079109dca5664cce6f317373782/...
1    www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...
2    serviciosbys.com/paypal.cgi.bin.get-into.herf....
3    mail.printakid.com/www.online.americanexpress....
4    thewhiskeydregs.com/wp-content/themes/widescre...
5                 smilesvoegol.servebbs.org/voegol.php
6    premierpaymentprocessing.com/includes/boleto-2...
7    myxxxcollection.com/v1/js/jih321/bpd.com.do/do...
8                                  super1000.info/docs
9    horizonsgallery.com/js/bin/ssl1/_id/www.paypal...
Name: URL, dtype: object


Contoh URL Good (legitimate):


18231    esxcc.com/js/index.htm?us.battle.net/noghn/en/...
18232    wwweira¯&nvinip¿ncH¯wVö%ÆåyDaHðû/ÏyEùuË\nÓ6...
18233    'www.institutocgr.coo/web/media/syqvem/dk-óij...
18234                    Yìê
koãÕ»Î§DéÎl½ñ¡ââqtò¸/à; Í
18236    ruta89fm.com/images/AS@Vies/1i75cf7b16vc<Fd16...
18237                                                ¾5092
18238    esxcc.com/js/index.htm?us.battle.net/login/en/...
18239    esxcc.com/js/index.htm?us.battle.net/login/en/...
18240                                   @
ÒÊ
\t¹Ë¨öí

18241         >Iq9E) l¬0ZÝìÃw!'Ôp£ãW§&£Ñ
p&$ImÞôöÝYÖ
Name: URL, dtype: object

PEMBERSIHAN


In [8]:
df.isnull().sum()

URL      0
Label    0
dtype: int64

In [9]:
df.duplicated().sum()

42150

In [10]:
# Panggil fungsi untuk menghapus baris duplikat dan pastikan agar perubahan disimpan kembali ke 'df'
df.drop_duplicates(inplace=True)

# Cek kembali dataset menggunakan duplicated().sum()
df.duplicated().sum()


0

## 8. Resolusi Konflik Label

URL yang sama bisa muncul lebih dari sekali dengan label berbeda (good & bad). Ini masalah serius — model tidak bisa belajar dari data yang kontradiktif. Kita hapus seluruh baris yang URL-nya memiliki label konflik.

In [11]:
# Cari URL yang punya lebih dari 1 label unik (konflik label)
label_conflict = df.groupby('URL')['Label'].nunique()
conflicting_urls = label_conflict[label_conflict > 1].index

print("Jumlah URL dengan label konflik:", len(conflicting_urls))
print("Jumlah baris yang akan dihapus:", df['URL'].isin(conflicting_urls).sum())

# Hapus seluruh baris yang URL-nya termasuk dalam daftar konflik
df = df[~df['URL'].isin(conflicting_urls)].reset_index(drop=True)

# Verifikasi tidak ada lagi konflik
print("Sisa konflik setelah cleaning:", (df.groupby('URL')['Label'].nunique() > 1).sum())
print("Shape setelah cleaning:", df.shape)

Jumlah URL dengan label konflik: 1
Jumlah baris yang akan dihapus: 2


Sisa konflik setelah cleaning: 0
Shape setelah cleaning: (507194, 2)


In [16]:
import re

def is_valid_url(url):
    """
    Validasi dasar untuk menyaring baris yang bukan URL sungguhan.
    Kriteria:
    1. Panjang minimal 7 karakter
    2. Mengandung setidaknya satu titik (.)
    3. Mengandung minimal 3 karakter alfanumerik berurutan
       (menyaring string yang isinya cuma simbol/karakter unicode acak)
    """
    if not isinstance(url, str):
        return False
    if len(url) < 7:
        return False
    if '.' not in url:
        return False
    if not re.search(r'[a-zA-Z0-9]{3,}', url):
        return False
    return True

# Terapkan validasi
before = df.shape[0]
valid_mask = df['URL'].apply(is_valid_url)

print("Jumlah baris tidak valid yang akan dihapus:", (~valid_mask).sum())
print("\nContoh baris yang akan dibuang:")
print(df[~valid_mask].head(10))

df = df[valid_mask].reset_index(drop=True)

print("\nShape sebelum:", before, "-> sesudah:", df.shape[0])
print("\nDistribusi label setelah full cleaning:")
print(df['Label'].value_counts())

Jumlah baris tidak valid yang akan dihapus: 114

Contoh baris yang akan dibuang:
                                                     URL Label
12011                             bin/webscr?cmd=_login-   bad
18232  wwweira¯&nvinip¿ncH¯wVö%ÆåyDaHðû/ÏyEùuË\nÓ6...  good
18234                  YìêkoãÕ»Î§DéÎl½ñ¡ââqtò¸/à; Í  good
18235                                   è kvüDâQÎé4¼Ò©   bad
18237                                              ¾5092  good
18240                                 @ÒÊ\t¹Ë¨öí  good
18241       >Iq9E) l¬0ZÝìÃw!'Ôp£ãW§&£Ñp&$ImÞôöÝYÖ  good
18244                        ê­~l+×û¢ïÄÁR"+íI  good
18245  RybjUxÙãl5»7ÆE%ÝÔk+h|U+ýk©ìÉ½Æq]âF·õ...  good
18248  ¶Ã·¼Øé¯·¤êZ}²^v³ÑÏ¯nD-³ª\áå½^oz¤8\t...  good

Shape sebelum: 507196 -> sesudah: 507082

Distribusi label setelah full cleaning:
Label
good    392832
bad     114250
Name: count, dtype: int64
